# MultimodalMU4Sum - Colab runner
Runtime GPU required. Data, checkpoints and outputs live on Google Drive so they survive disconnects; every script resumes from where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/<user>/MultimodalMU4Sum.git"  # <- set your repo
BRANCH = "main"
import os
if not os.path.exists('/content/MultimodalMU4Sum'):
    !git clone -b {BRANCH} {REPO_URL} /content/MultimodalMU4Sum
%cd /content/MultimodalMU4Sum
!git pull
!pip install -q -e ".[all]"

In [ ]:
import os
ROOT = "/content/drive/MyDrive/MU4Sum"
os.environ["MU4SUM_DATA_ROOT"] = f"{ROOT}/data"
os.environ["MU4SUM_OUTPUT_ROOT"] = f"{ROOT}/outputs"
# Kaggle download: put kaggle.json credentials (or KAGGLE_USERNAME / KAGGLE_KEY) here if the dataset requires them

## 1. Data pipeline (download -> process -> teacher augmentation)

In [ ]:
!python scripts/run_data_pipeline.py --dataset iu_xray --model qwen2_vl_7b

## 2. Fine-tuning (resumes automatically from the latest checkpoint)

In [ ]:
!python scripts/run_finetune.py --model qwen2_vl_7b --dataset iu_xray

## 3. Unlearning

In [ ]:
!python scripts/run_unlearn.py --model qwen2_vl_7b --dataset iu_xray --method grad_diff

## 4. Evaluation (per aspect)

In [ ]:
!python scripts/run_evaluate.py --model qwen2_vl_7b --dataset iu_xray --target finetune --unlearning grad_diff
!python scripts/run_evaluate.py --model qwen2_vl_7b --dataset iu_xray --target grad_diff

## 5. Sync back
On the local machine run `python scripts/sync_results.py --src <Drive>/MU4Sum/outputs` and commit the light artifacts (configs, metrics, predictions). Checkpoints are never copied.